# Проект "Секреты Темнолесья"

- Автор: Топорова Е.В.
- Дата: 09.02.2026

### Цель и задачи проекта

В этой тетрадке используем датасет `new_games.csv`, который содержит информацию о развитии игровой индустрии.

Цель: анализ развития игровой индустрии в период 2000-2013 годов для подготовки статьи, которая привлечет внимание любителей старых игр и заинтересует их проектом «Секреты Темнолесья».

Основные задачи:

1. Загрузить и проверить корректность исторических данных о видеоиграх.
2. Провести предобработку данных, включая обработку пропусков, дубликатов и преобразование типов.
3. Отфильтровать данные по периоду 2000-2013 годы.
4. Категоризовать игры по оценкам пользователей и экспертов.
5. Выявить топ-7 наиболее популярных игровых платформ.

### Описание данных

Датасет `new_games.csv` содержит информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
* `Name` — название игры.
* `Platform` — название платформы.
* `Year of Release` — год выпуска игры.
* `Genre` — жанр игры.
* `NA sales` — продажи в Северной Америке (в миллионах проданных копий).
* `EU sales` — продажи в Европе (в миллионах проданных копий).
* `JP sales` — продажи в Японии (в миллионах проданных копий).
* `Other sales` — продажи в других странах (в миллионах проданных копий).
* `Critic Score` — оценка критиков (от 0 до 100).
* `User Score` — оценка пользователей (от 0 до 10).
* `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержание проекта

[1. Загрузка данных и знакомство с ними](#1-загрузка-данных)

[2. Проверка ошибок в данных и их предобработка](#2-2-проверка-ошибок-и-предобработка)

[3. Фильтрация данных](#3-фильтрация)

[4. Категоризация данных](#4-категоризация)

[5. Итоговый вывод](#5-вывод)

---

<a id="1-загрузка-данных"></a>
## 1 Загрузка данных и знакомство с ними

Загрузим необходимые библиотеки Python.


In [135]:
import pandas as pd

Загрузим данные датасета `new_games.csv`.

In [136]:
df = pd.read_csv('new_games.csv')

Выведем первые строки датасета.


In [137]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Изучим информацию о датасете.


In [138]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


### Вывод 
Данные соотвествуют заявленному описанию, они состоят из 16 956 строк и 11 столбцов.

В 6 столбцах из 11 присутствуют пропуски.

У некоторых данных неверно определен тип:
- Год логичнее представить в формате int64, а не float64;
- EU sales, JP sales должны быть в формате float64;
- Оценки пользователей также должны быть в float64.

Для удобства и красоты написания кода названия столбцов можно сделать в формате snake_case.

---
<a id="2-проверка-ошибок-и-предобработка"></a>
## 2 Проверка ошибок в данных и их предобработка


### 2.1 Названия столбцов датафрейма

Выведем текущие названия столбцов.

In [139]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')

Приведем названия столбцов к формату snake_case, так как с ним удобнее работать.

In [140]:
# Приводим все названия к нижнему регистру
df.columns = df.columns.str.lower()

# Заменяем пробелы в названиях на нижнее подчеркивание
df.columns = df.columns.str.replace(' ', '_')

# Выводим результат
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')

### 2.2 Типы данных

Как было сказано ранее, у некоторых столбцов неверно определен тип:
- eu_sales, jp_sales и user_score были определены как object вместо float64, так как в них присутствуют строковые значения;
- year_of_release определен как float вместо int, так как содержит пропуски. Сначала избавимся от пропусков, а затем преобразуем тип (см. пункт 2.3).

Проведем преобразование типов.

In [141]:
# Переводим 'eu_sales', 'jp_sales', 'user_score' в float64, при этом непреобразуемые значения заменяем на NaN (errors='coerce')
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce').astype('float64')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce').astype('float64')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce').astype('float64')

# Проверяем преобразования
df.dtypes

name                   str
platform               str
year_of_release    float64
genre                  str
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating                 str
dtype: object

### 2.3 Наличие пропусков в данных

Посчитаем количество пропусков в каждом столбце в абсолютных значениях.

In [142]:
df.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

Посчитаем количество пропусков в каждом столбце в относительных значениях.

In [143]:
df.isna().sum() / df.shape[0] * 100

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

Пропущенные значения наблюдаются в:
- name - 0.01% пропусков,
- year_of_release - 1.62% пропусков,
- genre - 0.01% пропусков,
- critic_score - 51.39% пропусков,
- user_score - 40.12% пропусков,
- rating - 40.52% пропусков.

Возможные причины пропусков:
- name и genre - скорее всего, техническая ошибка при сборе данных;
- в critic_score пропуски могли возникнуть из-за того, что  некоторые игры непопулярны и не привлекли внимание критиков;
- user_score - опять же, непопулярные игры с маленькой аудиторией;
- rating - игры, не получившие рейтинга организации ESRB;
- year_of_release - техническая ошибка.

Стратегия работы с пропусками:
1. В столбцах name и genre пропуски составляют менее 1%, поэтому их можно удалить.
2. В столбцах critic_score и user_score пропусков довольно много (51 и 40%), кроме того, эти данные понадобятся при разделении игр на категории. Можно заполнить пропуски индикаторами, а в дальнейшей работе выделить отдельную категорию 'нет оценки'.
3. Значения rating в данном проекте нам не понадобятся, поэтому оставим столбец без изменений.
4. year_of_release - можно для начала проверить, у каких игр не хватает года, а затем решить, что с ними делать.


In [144]:
# Сохраняем количество строк до изменений
total_rows = df.shape[0]

# Удаляем пропуски в name и genre
df = df.dropna(subset=['name', 'genre'])

# Посмотрим, у каких игр пропущен год
df[df['year_of_release'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
183,Madden NFL 2004,PS2,NaN,Sports,4.26,0.26,0.01,0.71,94.0,8.5,E
379,FIFA Soccer 2004,PS2,NaN,Sports,0.59,2.36,0.04,0.51,84.0,6.4,E
458,LEGO Batman: The Videogame,Wii,NaN,Action,1.80,0.97,0.00,0.29,74.0,7.9,E10+
477,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,1.57,1.02,0.00,0.41,NaN,NaN,NaN
611,Space Invaders,2600,NaN,Shooter,2.36,0.14,0.00,0.03,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16609,PDC World Championship Darts 2008,PSP,NaN,Sports,0.01,0.00,0.00,0.00,43.0,NaN,E10+
16641,Freaky Flyers,GC,NaN,Racing,0.01,0.00,0.00,0.00,69.0,6.5,T
16685,Inversion,PC,NaN,Shooter,0.01,0.00,0.00,0.00,59.0,6.7,M
16695,Hakuouki: Shinsengumi Kitan,PS3,NaN,Adventure,0.01,0.00,0.00,0.00,NaN,NaN,NaN


Можно заметить, что у некоторых игр год прописан прямо в названии. Можем воспользоваться этим и восстановить некоторые данные. Остальные данные можем оставить без именений, так как пропущенные значения отбросятся при фильтрации.

In [145]:
def extract_year_from_name(row):
    """
    Извлекает год из названия игры.
    Проверяет последние 4 символа.
    """  
    if pd.isna(row['year_of_release']):
        last_4 = row['name'][-4:]
        if last_4.isdigit():
            year = int(last_4)
            # Проверяем, что это реалистичный год
            if df['year_of_release'].min() <= year <= df['year_of_release'].max():
                return year
        return None
    else:
        return row['year_of_release']


df['year_of_release'] = df.apply(extract_year_from_name, axis=1)
# Удаляем оставшиеся пропущенные значения
df = df.dropna(subset=['year_of_release'])
# Переводим 'year_of_release' в int64
df['year_of_release'] = df['year_of_release'].astype('int64')

### 2.4 Явные и неявные дубликаты в данных

Проверим наличие неявных дубликатов в столбце platform.

In [146]:
df['platform'].unique()

<StringArray>
[ 'Wii',  'NES',   'GB',   'DS', 'X360',  'PS3',  'PS2', 'SNES',  'GBA',
  'PS4',  '3DS',  'N64',   'PS',   'XB',   'PC', '2600',  'PSP', 'XOne',
 'WiiU',   'GC',  'GEN',   'DC',  'PSV',  'SAT',  'SCD',   'WS',   'NG',
 'TG16',  '3DO',   'GG', 'PCFX']
Length: 31, dtype: str

Неявные дубликаты в platform отсутствуют.

Проверим наличие неявных дубликатов в столбце genre.

In [147]:
df['genre'].unique()

<StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy',
         'MISC', 'ROLE-PLAYING',       'RACING',       'ACTION',
      'SHOOTER',     'FIGHTING',       'SPORTS',     'PLATFORM',
    'ADVENTURE',   'SIMULATION',       'PUZZLE',     'STRATEGY']
Length: 24, dtype: str

Столбец genre содержит неявные дубликаты из-за записей с разным регистром. Чтобы устранить эту проблему, приведем все данные к нижнему регистру.

In [148]:
# Приводим genre к нижнему регистру
df['genre'] = df['genre'].str.lower()

# Выводим результат
df['genre'].unique()

<StringArray>
[      'sports',     'platform',       'racing', 'role-playing',
       'puzzle',         'misc',      'shooter',   'simulation',
       'action',     'fighting',    'adventure',     'strategy']
Length: 12, dtype: str

Проверим наличие неявных дубликатов в столбце rating.

In [149]:
df['rating'].unique()

<StringArray>
['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Length: 9, dtype: str

Так как K-A это устаревшее название категории E, заменим все вхождения K-A на E.

In [150]:
df['rating'] = df['rating'].replace('K-A', 'E')
df['rating'].unique()

<StringArray>
['E', nan, 'M', 'T', 'E10+', 'AO', 'EC', 'RP']
Length: 8, dtype: str

Проверим наличие явных дубликатов в данных.

In [151]:
df.duplicated().sum()

np.int64(235)

В данных обнаружено 235 дубликатов. Избавимся от них путем удаления.

In [152]:
# Сортируем датафрейм по всем столбцам
df = df.sort_values(by=list(df.columns))

# Все повторяющиеся строки, кроме первой, удаляются
df = df.drop_duplicates()

Посчитаем количество удалённых строк в абсолютном и относительном значениях.

In [153]:
# Количество строк после изменений
final_row_count = df.shape[0]

# Количество удаленных строк в абсолютном и относительном значениях
count_abs = total_rows - final_row_count
count_rel = (total_rows - final_row_count) / total_rows * 100

count_abs, count_rel

(497, 2.9311158292050012)

### 2.5 Вывод

В ходе предобработки данных:
1. Названия столбцов были приведены к более удобному формату snake_case.
2. У столбцов eu_sales, jp_sales и user_score и year_of_release изменены типы.
3. В name, genre и year_of_release удалены пропуски.
4. Устранены неявные в столбцах genre и rating, а также явные дубликаты в датафрейме.

В результате преобразований датафрейм сократился на 497 строки (2.93%), что приемлемо.

---
<a id="3-фильтрация"></a>
## 3 Фильтрация данных

Отберем данные за период с 2000 по 2013 год включительно и сохраним в новый датафрейм `df_actual`.

In [154]:
df_actual = df[df['year_of_release'].between(2000, 2013)]

---
<a id="4-категоризация"></a>
## 4 Категоризация данных
    
Проведем категоризацию данных:
- Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [155]:
def categorize_user(score):
    """
    Категоризация пользовательских оценок от 0 до 10
    с учетом пропущенных значений.
    """
    if 8 <= score <= 10:
        return 'высокая оценка'
    elif 3 <= score < 8:
        return 'средняя оценка'
    elif 0 <= score < 3:
        return 'низкая оценка'
    else:
        return 'нет оценки'

df_actual['cat_by_user_score'] = df_actual['user_score'].apply(categorize_user)

df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,cat_by_user_score
3394,Frozen: Olaf's Quest,3DS,2013,platform,0.27,0.27,0.00,0.05,NaN,NaN,NaN,нет оценки
3906,Frozen: Olaf's Quest,DS,2013,platform,0.21,0.26,0.00,0.04,NaN,NaN,NaN,нет оценки
2478,Tales of Xillia 2,PS3,2012,role-playing,0.20,0.12,0.45,0.07,71.0,7.9,T,средняя оценка
8460,.hack//G.U. Vol.1//Rebirth,PS2,2006,role-playing,0.00,0.00,0.17,0.00,NaN,NaN,NaN,нет оценки
7182,.hack//G.U. Vol.2//Reminisce,PS2,2006,role-playing,0.11,0.09,0.00,0.03,NaN,NaN,NaN,нет оценки
...,...,...,...,...,...,...,...,...,...,...,...,...
647,uDraw Studio,Wii,2010,misc,1.65,0.57,0.00,0.20,71.0,NaN,E,нет оценки
8397,uDraw Studio: Instant Artist,Wii,2011,misc,0.06,0.09,0.00,0.02,NaN,NaN,E,нет оценки
15836,uDraw Studio: Instant Artist,X360,2011,misc,0.01,0.01,0.00,0.00,54.0,5.7,E,средняя оценка
477,wwe Smackdown vs. Raw 2006,PS2,2006,fighting,1.57,1.02,0.00,0.41,NaN,NaN,NaN,нет оценки


- Разделим все игры по оценкам критиков и выделим такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [156]:
def categorize_critic(score):
    """
    Категоризация оценок критиков от 0 до 100 с учетом пропущенных значений.
    """
    if 80 <= score <= 100:
        return 'высокая оценка'
    elif 30 <= score < 80:
        return 'средняя оценка'
    elif 0 <= score < 30:
        return 'низкая оценка'
    else:
        return 'нет оценки'

df_actual['cat_by_critic_score'] = df_actual['critic_score'].apply(categorize_critic)

df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,cat_by_user_score,cat_by_critic_score
3394,Frozen: Olaf's Quest,3DS,2013,platform,0.27,0.27,0.00,0.05,NaN,NaN,NaN,нет оценки,нет оценки
3906,Frozen: Olaf's Quest,DS,2013,platform,0.21,0.26,0.00,0.04,NaN,NaN,NaN,нет оценки,нет оценки
2478,Tales of Xillia 2,PS3,2012,role-playing,0.20,0.12,0.45,0.07,71.0,7.9,T,средняя оценка,средняя оценка
8460,.hack//G.U. Vol.1//Rebirth,PS2,2006,role-playing,0.00,0.00,0.17,0.00,NaN,NaN,NaN,нет оценки,нет оценки
7182,.hack//G.U. Vol.2//Reminisce,PS2,2006,role-playing,0.11,0.09,0.00,0.03,NaN,NaN,NaN,нет оценки,нет оценки
...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,uDraw Studio,Wii,2010,misc,1.65,0.57,0.00,0.20,71.0,NaN,E,нет оценки,средняя оценка
8397,uDraw Studio: Instant Artist,Wii,2011,misc,0.06,0.09,0.00,0.02,NaN,NaN,E,нет оценки,нет оценки
15836,uDraw Studio: Instant Artist,X360,2011,misc,0.01,0.01,0.00,0.00,54.0,5.7,E,средняя оценка,средняя оценка
477,wwe Smackdown vs. Raw 2006,PS2,2006,fighting,1.57,1.02,0.00,0.41,NaN,NaN,NaN,нет оценки,нет оценки


- Проверим результат категоризации: сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [157]:
user_score_series = df_actual['cat_by_user_score'].value_counts()
user_score_df = user_score_series.reset_index()
user_score_df.columns = ['Категория оценки пользователя', 'Количество игр']
user_score_df

,Категория оценки пользователя,Количество игр
0,нет оценки,6304
1,средняя оценка,4083
2,высокая оценка,2293
3,низкая оценка,116


In [158]:
critic_series = df_actual['cat_by_critic_score'].value_counts()
critic_df = critic_series.reset_index()
critic_df.columns = ['Категория оценки критика', 'Количество игр']
critic_df

,Категория оценки критика,Количество игр
0,нет оценки,5616
1,средняя оценка,5427
2,высокая оценка,1698
3,низкая оценка,55


- Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [159]:
df_actual['platform'].value_counts().reset_index().head(7)

,platform,count
0,PS2,2134
1,DS,2121
2,Wii,1275
3,PSP,1181
4,X360,1123
5,PS3,1087
6,GBA,811


---
<a id="5-вывод"></a>
## 5 Итоговый вывод

Исходный датасет `new_games.csv` содержал 16956 записей и 11 столбцов.

Была проведена следующая предобработка:
1. Названия столбцов приведены к snake_case.
2. Преобразование типов:
- Преобразованы столбцы продаж (eu_sales, jp_sales) из object со значениями 'unknown' в float64.
- Преобразован year_of_release в int64 с предварительной обработкой пропусков.
- Обработан user_score: удалены значения 'tbd', преобразован в float64.
3. Обработка пропусков:
- year_of_release (1.62%), name и genre (0.01%): строки удалены из-за незначительного количества.
- critic_score (51.39%), user_score (40.12%), rating (40.52%): оставлены как есть.
4. Обработка дубликатов:
- устранены неявные дубликаты в genre,
- объединен старый рейтинг 'K-A' с 'E' (Everyone),
- удалены явные дубликаты.

В результате предобработки датасет сократился на 2.93%.

Далее был создан срез данных df_actual, содержащий записи за 2000-2013 годы включительно. В него вошло 12796 записей.

В этот срез были добавлены новые столбцы:

cat_by_user_score - категория пользовательской оценки (шкала 0-10):
- высокая оценка: 8-10 баллов (включая 10),
- средняя оценка: 3-8 баллов (не включая 8),
- низкая оценка: 0-3 балла (не включая 3),
- нет оценки: для пропусков.

cat_by_critic_score - категория экспертной оценки (шкала 0-100):
- высокая оценка: 80-100 баллов (включая 100),
- средняя оценка: 30-80 баллов (не включая 80),
- низкая оценка: 0-30 баллов (не включая 30),
- нет оценки: для пропусков.